# [LAB 05] SHAP 값 → Tableau 시각화용 CSV 생성

- **목적**: 4개 Channel 분류 모델(LogisticRegression, RandomForest, SGDClassifier, XGBoost)에 대해 SHAP 값을 계산하고,
  Tableau에서 SHAP summary(beeswarm scatter) 형태로 그리기 위한 `shap_values_for_tableau.csv` 생성.
- **Tableau 설정**: Rows → feature + jitter, Columns → shap_value, Color → feature_value, Detail → customer_id, Filter → model.

## 1. 라이브러리 및 경로 설정

In [1]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import shap
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from hossam import load_data

LAB05_DIR = Path('.').resolve()
RAW_COLS = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
log_cols = [f"log_{c}" for c in RAW_COLS]
FEATURE_COLS = log_cols + ["Region_2", "Region_3"]  # 8개 피처

c:\Users\bohee\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📦 아이티윌 이광호 강사가 제작한 라이브러리를 사용중입니다.
📚 자세한 사용 방법은 https://py.hossam.kr 을 참고하세요.
📧 Email: leekh4232@gmail.com
🎬 Youtube: https://www.youtube.com/@hossam-codingclub
📝 Blog: https://blog.hossam.kr/
🔖 Version: 0.4.8


## 2. 데이터 로드 (08·00과 동일 전처리)

In [2]:
def get_data_and_split():
    """노트북·00 스크립트와 동일한 전처리·split. Region은 범주형 원핫(Region_2, Region_3)."""
    origin = load_data("wholesale_customers")
    df = origin[RAW_COLS + ["Region", "Channel"]].copy()
    df_log = df[RAW_COLS].copy()
    for c in RAW_COLS:
        df_log[f"log_{c}"] = np.log1p(df_log[c])
    region_dummies = pd.get_dummies(df["Region"], prefix="Region", drop_first=True)
    df_log = pd.concat([df_log, region_dummies], axis=1)
    scaler_path = LAB05_DIR / "wholesale_scaler.pkl"
    if not scaler_path.exists():
        scaler = StandardScaler()
        X_all = scaler.fit_transform(df_log)
        col_ix = [df_log.columns.get_loc(c) for c in FEATURE_COLS]
        X = X_all[:, col_ix]
    else:
        with open(scaler_path, "rb") as f:
            scaler = pickle.load(f)
        X_all = scaler.transform(df_log)
        col_ix = [df_log.columns.get_loc(c) for c in FEATURE_COLS]
        X = X_all[:, col_ix]
    y = (df["Channel"] - 1).values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=52, stratify=y
    )
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = get_data_and_split()
X = X_test  # SHAP 계산은 테스트 세트 기준 (Tableau 시각화용)
n_samples, n_features = X.shape
print("X (테스트 세트):", X.shape)
print("Features:", FEATURE_COLS)

이 데이터 세트는 도매 유통업체의 고객 정보를 담고 있습니다. 다양한 제품 카테고리에 대한 연간 지출액(mu, 화폐 단위)을 포함합니다. (출처: https://www.kaggle.com/datasets/binovi/wholesale-customers-data-set)

컬럼명            의미                  설명
----------------  --------------------  --------------------------------------------------------------------------------------------
Channel           유통 채널             고객의 거래 채널을 나타냄. 주로 Horeca(호텔·레스토랑·카페) 또는 Retail(소매점) 구분에 사용됨
Region            지역                  고객이 속한 지리적 지역 구분 변수. 특정 국가 내의 권역 정보
Fresh             신선식품 구매액       육류, 채소, 과일 등 신선식품 카테고리에 대한 연간 구매 금액
Milk              유제품 구매액         우유, 치즈, 요거트 등 유제품 카테고리에 대한 연간 구매 금액
Grocery           식료품 구매액         가공식품, 일반 식료품 등 장기 보관 식품 카테고리 구매 금액
Frozen            냉동식품 구매액       냉동 육류, 냉동 가공식품 등 냉동식품 카테고리 구매 금액
Detergents_Paper  세제·종이류 구매액    세제, 화장지, 키친타월 등 생활 소모품 구매 금액
Delicassen        즉석·가공식품 구매액  즉석식품, 델리 식품, 가공 반찬류 등의 구매 금액

X (테스트 세트): (110, 8)
Features: ['log_Fresh', 'log_Milk', 'log_Grocery', 'log_Frozen', 'log_Detergents_Paper', 

## 3. 모델 로드 (4개)

In [3]:
def load_channel_models(lab05_dir: Path):
    """학습된 4개 Channel 분류 모델을 pkl에서 로드하여 dictionary로 반환."""
    mapping = {
        "LogisticRegression": "wholesale_logistic.pkl",
        "RandomForest": "wholesale_rf.pkl",
        "SGDClassifier": "wholesale_sgd.pkl",
        "XGBoost": "wholesale_xgb.pkl",
    }
    models = {}
    for name, pkl_name in mapping.items():
        path = lab05_dir / pkl_name
        if not path.exists():
            print(f"경고: {pkl_name} 없음, 스킵.")
            continue
        with open(path, "rb") as f:
            models[name] = pickle.load(f)
    return models

# 4개 모델만 사용 (pkl 없으면 해당 모델 스킵)
MODEL_NAMES = ["LogisticRegression", "RandomForest", "SGDClassifier", "XGBoost"]
loaded = load_channel_models(LAB05_DIR)
models = {k: loaded[k] for k in MODEL_NAMES if k in loaded}
display(pd.Series(list(models.keys()), name="model"))

0    LogisticRegression
1          RandomForest
2         SGDClassifier
3               XGBoost
Name: model, dtype: object

## 4. 모델별 SHAP 계산 (TreeExplainer / LinearExplainer)

In [4]:
TREE_MODELS = ["RandomForest", "XGBoost"]  # TreeExplainer 사용
LINEAR_MODELS = ["LogisticRegression", "SGDClassifier"]  # LinearExplainer 사용

def compute_shap_values(model, model_name: str, X_background, X_eval, feature_names):
    """
    모델 타입에 따라 explainer를 선택하여 SHAP 값을 계산.
    이진 분류 시 shap_values가 list면 양성 클래스(sv[1]) 사용.
    """
    if model_name in TREE_MODELS:
        explainer = shap.TreeExplainer(model, X_background)
    else:
        explainer = shap.LinearExplainer(model, X_background)
    sv = explainer.shap_values(X_eval)
    if isinstance(sv, list):
        sv = sv[1]  # 이진 분류 양성 클래스
    sv = np.asarray(sv)
    # 3차원 (n_samples, n_features, n_classes) → 양성 클래스만 (n_samples, n_features)
    if sv.ndim == 3:
        sv = sv[:, :, 1]
    return sv

shap_arrays = {}  # model_name -> (n_samples, n_features)
for name, clf in models.items():
    try:
        sv = compute_shap_values(clf, name, X_train, X, FEATURE_COLS)
        shap_arrays[name] = sv
        print(f"{name}: SHAP shape {sv.shape}")
    except Exception as e:
        print(f"{name}: 오류 - {e}")

display(pd.DataFrame({
    "model": list(shap_arrays.keys()),
    "shape": [str(shap_arrays[k].shape) for k in shap_arrays],
}))

LogisticRegression: SHAP shape (110, 8)
RandomForest: SHAP shape (110, 8, 2)
SGDClassifier: SHAP shape (110, 8)
XGBoost: SHAP shape (110, 8)


,model,shape
0,LogisticRegression,"(110, 8)"
1,RandomForest,"(110, 8, 2)"
2,SGDClassifier,"(110, 8)"
3,XGBoost,"(110, 8)"


## 5. SHAP 결과를 long format으로 변환 + feature_value, shap_abs, feature_importance, jitter

In [5]:
def shap_wide_to_long(
    model_name: str,
    shap_matrix: np.ndarray,
    X_values: np.ndarray,
    feature_names: list,
    rng: np.random.Generator,
):
    """
    (n_samples, n_features) SHAP 행렬을 long format DataFrame으로 변환.
    컬럼: model, customer_id, feature, shap_value, feature_value, shap_abs, feature_importance, jitter.
    """
    n_samples, n_features = shap_matrix.shape
    rows = []
    for i in range(n_samples):
        for j in range(n_features):
            sh = float(shap_matrix[i, j])
            fv = float(X_values[i, j])
            rows.append({
                "model": model_name,
                "customer_id": i,
                "feature": feature_names[j],
                "shap_value": sh,
                "feature_value": fv,
                "shap_abs": abs(sh),
            })
    df = pd.DataFrame(rows)
    # feature 별 평균 |SHAP| → feature_importance
    fi = df.groupby("feature")["shap_abs"].transform("mean")
    df["feature_importance"] = fi
    # Tableau beeswarm용 jitter (같은 행에 동일 jitter 적용 시 feature별로 다르게 할지 결정)
    # 요구: feature + jitter 를 Rows에 쓰므로, (customer_id, feature)당 하나의 jitter
    df["jitter"] = rng.uniform(-0.2, 0.2, size=len(df))
    return df

rng = np.random.default_rng(52)
all_dfs = []
for name, sv in shap_arrays.items():
    df_long = shap_wide_to_long(name, sv, X, FEATURE_COLS, rng)
    all_dfs.append(df_long)

shap_long = pd.concat(all_dfs, ignore_index=True)
display(shap_long.head(10))

ValueError: too many values to unpack (expected 2)

## 6. CSV 저장 (utf-8-sig)

In [ ]:
COL_ORDER = [
    "model", "customer_id", "feature", "shap_value", "feature_value",
    "shap_abs", "feature_importance", "jitter",
]
out_path = LAB05_DIR / "shap_values_for_tableau.csv"
shap_long[COL_ORDER].to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"SHAP CSV 저장 완료: {out_path.resolve()}")
display(shap_long.groupby("model").size().to_frame(name="rows"))